# 02 · 进阶：视频分析交互体验

先完成 [API 基础版](01_api_quickstart.ipynb)，再用本页面体验同样的协议。支持持续描述、主动检测、同一会话追问，以及描述＋检测双路并行。

页面把基础版的调用收进交互组件：Notebook 准备配置，`flow_demo.py` 管理按钮和会话组合，`flow_api.py` 管理单个 API 会话。先读[进阶版工作原理](../docs/interactive_playground.md)，可对照按钮与基础版代码。

依次运行第 1–4 步，**点击开始才会调用模型**。第 3 步会上传视频。第 5 步为备用停止入口，第 6 步为可选诊断。

首次安装：`python -m pip install -r examples/requirements.txt`，选择 **Python (VLX-Flow Demos)** 内核。[Realtime 文档](https://s.apifox.cn/46c7cae8-f2d7-4780-a960-e531d6c5eaa8/9598749m0)

## 1 · 检查环境与加载组件

In [ ]:
from pathlib import Path
import asyncio
import hashlib
import json
import sys
import time
from getpass import getpass
from urllib.parse import urlsplit, urlunsplit
from importlib.metadata import version

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "examples/flow_api.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("请从 VLX-Flow 仓库内打开 Notebook。")
if str(PROJECT_ROOT / "examples") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "examples"))

import httpx
from websockets.asyncio.client import connect
from websockets.exceptions import ConnectionClosed
from IPython.display import display
from flow_api import RealtimeSession, api_root, verify_upload

# 保留已创建的对象，重复运行时不会丢失停止入口。
_api_sessions = globals().setdefault("_api_sessions", [])
for package in ("httpx", "websockets", "python-socks", "ipykernel"):
    print(f"{package}: {version(package)}")
print("环境就绪；尚未上传或连接模型。")

from flow_demo import InteractivePlayground, DEFAULT_SETTINGS, DESCRIPTION_INSTRUCTION, DETECTION_INSTRUCTION
print("anywidget:", version("anywidget"))

## 2 · 鉴权、视频与初始参数

模型、分析参数、两套指令和时限均可在页面修改。视频使用相对路径；令牌只保存在 Python 内存。

In [ ]:
if globals().get("playground") is not None and (playground.running or playground.cleanup_blocked):
    raise RuntimeError("请先运行停止单元格。")

if any(s.running or s.cleanup_blocked for s in _api_sessions):
    raise RuntimeError("请先运行停止单元格，关闭旧会话后再修改配置。")

API_ENDPOINT = api_root(input("平台 API 根地址（https://…）："))
API_KEY = getpass("访问令牌：").strip()
if not API_KEY:
    raise ValueError("访问令牌不能为空。")

VIDEO_PATH = PROJECT_ROOT / "examples/assets/videos/vlxflow-video04_keyframe_1s.mp4"
MODEL_ID = "VLX-Flow-1.5-10B"
INSTRUCTION = "持续观看视频画面, 并描述画面内容"
PARAMETERS = {
    "interval_seconds": 1000,  # 文档规定为毫秒，1000 即 1 秒
    "batch_size": 2,          # 每累计 2 张新帧触发分析；支持 1 或 2
    "window_size": 8,         # 窗口数，支持 1–8
    "temperature": 0.2,       # 0 <= 值 < 2
    "max_tokens": 32,         # 每轮 1–32 tokens
}
DURATION_SECONDS = 60  # 本地自动停止时限，不放入 API 请求
USE_PROXY = True       # 使用系统代理；如需直连改为 False
if not VIDEO_PATH.is_file():
    raise FileNotFoundError(VIDEO_PATH.name)
print("视频：", VIDEO_PATH.relative_to(PROJECT_ROOT))
print("指令：", INSTRUCTION)
print("参数：", PARAMETERS)

INITIAL_SETTINGS = {**DEFAULT_SETTINGS, "model": MODEL_ID, "duration_seconds": DURATION_SECONDS, **PARAMETERS}
DESCRIPTION = INSTRUCTION
DETECTION = DETECTION_INSTRUCTION

## 3 · 上传视频

只上传，不启动模型；同一内核可复用结果。

In [ ]:
if globals().get("playground") is not None and (playground.running or playground.cleanup_blocked):
    raise RuntimeError("请先停止旧会话。")

if any(s.running or s.cleanup_blocked for s in _api_sessions):
    raise RuntimeError("请先关闭旧会话，再更换或上传视频。")
FORCE_UPLOAD = False
stat = VIDEO_PATH.stat()
upload_identity = (API_ENDPOINT, hashlib.sha256(API_KEY.encode()).hexdigest(),
                   str(VIDEO_PATH), stat.st_size, stat.st_mtime_ns)
upload_cache = globals().setdefault("_flow_upload_cache", {})
if not FORCE_UPLOAD and upload_identity in upload_cache:
    VIDEO_URL = upload_cache[upload_identity]
    print("复用当前内核的上传结果。")
else:
    VIDEO_URL = None
    async with httpx.AsyncClient(timeout=httpx.Timeout(300, connect=30), trust_env=USE_PROXY) as client:
        with VIDEO_PATH.open("rb") as video:
            response = await client.post(
                f"{API_ENDPOINT}/os/uploadFile",
                headers={"Authorization": f"Bearer {API_KEY}"},
                files={"file": (VIDEO_PATH.name, video, "video/mp4")},
                data={"type": "video/mp4"},
            )
    VIDEO_URL = verify_upload(response)  # 读取并校验 data.fileUrl
    upload_cache[upload_identity] = VIDEO_URL
    print("上传成功。")
UPLOADED_IDENTITY = upload_identity
print("URL 仅保存在内存中，尚未调用模型。")

## 4 · 打开交互页面

**本块首次运行只创建并显示组件，不启动模型。** `InteractivePlayground(...)` 读取本地视频并准备控件，`display(playground)` 显示页面；点击「开始分析」后才创建 API 会话。重复运行时会先关闭旧实例中的会话。

| 页面操作 | 对应基础版 | 协议行为 |
| --- | --- | --- |
| 开始分析 | 启动单元格 | 建立连接并发送 `session.create` |
| 发送追问 | 填写问题后运行追问单元格 | 原连接发送 `response.create` |
| 全部停止 | 停止单元格 | 请求关闭各会话并确认清理 |

选择持续任务，再确认参数与指令。描述与检测都是 `session.create.data.instruction`，不是额外的 API 模式参数。双路共用采样与生成参数，使用两套指令和两个独立会话；一个 URL 可被两路复用。两路独立处理和返回结果，不保证逐帧同步。

- **追问**：首轮分析到达后可发送；双路时明确选择目标会话。每路同一时间只提交一条未完成追问，结果与持续分析分开显示。
- **全部停止**：关闭所有已启动会话；到设定时限也会自动关闭。若一路异常，会停止另一条活动会话。关闭待确认时先检查关闭。
- **调参**：停止后直接修改控件，再开始，无需重新上传。重新运行本单元格则恢复第 2 步初始值。
- **预览**：可展开查看创建 JSON（隐藏视频 URL）。这展示客户端参数，不证明服务端如何执行指令。
- **视频时间**：本地预览不控制服务端进度，拖动或暂停视频不会把追问定位到该帧。服务端相对时间尚未验证等同于视频时间。回答旁的时间属于最近展示的有效结果；检测没有新有效内容时会保留旧时间，不能将两路标签差值直接解释为处理进度差。
- **检测效果**：当前页面折叠空文本/省略号但保留计数，不能把折叠等同于模型静默；是否触发告警需用实际视频测试。

In [ ]:
stat = VIDEO_PATH.stat()
identity = (API_ENDPOINT, hashlib.sha256(API_KEY.encode()).hexdigest(),
            str(VIDEO_PATH), stat.st_size, stat.st_mtime_ns)
if identity != UPLOADED_IDENTITY or not VIDEO_URL:
    raise RuntimeError("文件或鉴权已变化，请重新执行上传步骤。")

if globals().get("playground") is not None:
    await playground.shutdown()
    playground.close()
playground = InteractivePlayground(
    endpoint=API_ENDPOINT, api_key=API_KEY, video_path=VIDEO_PATH, video_url=VIDEO_URL,
    use_proxy=USE_PROXY, settings=INITIAL_SETTINGS, description=DESCRIPTION, detection=DETECTION,
)
display(playground)

## 5 · 全部停止（备用入口）

页面按钮与这里都能关闭所有会话。结束前必须确认关闭；不需要两种方式都运行。

In [ ]:
if globals().get("playground") is not None:
    await playground.shutdown()
print("无待确认的活动会话。")

## 6 · 事件诊断（可选）

按路显示最近事件；不打印密钥、请求头或 URL。可能包含视频描述与问题内容，分享前请检查。

In [ ]:
if globals().get("playground") is not None:
    events = {name: rows[-20:] for name, rows in playground.diagnostic_events().items()}
    print(json.dumps(events, ensure_ascii=False, indent=2))

提交前先停止会话，清除全部输出与保存的 Widget State 再保存。Notebook 和辅助文件仅包含公开 API 调用与展示代码。